
# HW2 — UniProt & ENSEMBL API

This notebook implements:

- `get_uniprot()`
- `uniprot_parse_response()`
- `get_ensembl()`
- `ensembl_parse_response()`
- `main()`

The notebook queries:

- UniProt (protein sequences)
- ENSEMBL (nucleotide / gene information)

All functions return parsed information as a pandas DataFrame.


In [ ]:

import requests
import pandas as pd
import re


## UniProt Functions

In [ ]:

def get_uniprot(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    response = requests.get(url)
    return response


In [ ]:

def uniprot_parse_response(resp):
    try:
        data = resp.json()
        accession = data.get("primaryAccession", None)

        organism = data.get("organism", {}).get("scientificName", None)

        geneInfo = data.get("genes", None)

        sequenceInfo = data.get("sequence", None)

        result = {
            accession: {
                "organism": organism,
                "geneInfo": geneInfo,
                "sequenceInfo": sequenceInfo,
                "type": "protein"
            }
        }

        return result

    except Exception as e:
        return {"error": str(e)}


## ENSEMBL Functions

In [ ]:

def get_ensembl(id):
    url = f"https://rest.ensembl.org/lookup/id/{id}?content-type=application/json"
    response = requests.get(url)
    return response


In [ ]:

def ensembl_parse_response(resp):
    try:
        data = resp.json()

        result = {
            data.get("id"): {
                "object_type": data.get("object_type"),
                "assembly_name": data.get("assembly_name"),
                "species": data.get("species"),
                "db_type": data.get("db_type"),
                "biotype": data.get("biotype"),
                "display_name": data.get("display_name"),
                "id": data.get("id"),
                "description": data.get("description"),
                "canonical_transcript": data.get("canonical_transcript"),
                "source": data.get("source")
            }
        }

        return result

    except Exception as e:
        return {"error": str(e)}


## Main Function

In [ ]:

def main(ids):

    results = {}

    uniprot_regex = r'^[A-NR-Z0-9]{6,10}$'
    ensembl_regex = r'^ENS[A-Z0-9]+'

    for id in ids:

        if re.match(uniprot_regex, id):
            resp = get_uniprot(id)

            if resp.status_code == 200:
                parsed = uniprot_parse_response(resp)
                results.update(parsed)
            else:
                results[id] = "error: invalid uniprot id"

        elif re.match(ensembl_regex, id):
            resp = get_ensembl(id)

            if resp.status_code == 200:
                parsed = ensembl_parse_response(resp)
                results.update(parsed)
            else:
                results[id] = "error: invalid ensembl id"

        else:
            results[id] = "error:unknown database"

    return pd.DataFrame(results).T


## Testing

In [ ]:

test_ids = ['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618']

main(test_ids)
